In [13]:
import pandas as pd
import numpy as np
import sqlite3
import os

In [14]:
os.chdir(r"C:\Users\Haider\Downloads\project\FARS2023NationalCSV")
conn = sqlite3.connect("accident_project.db")

In [15]:
tables = [
    "accident_project",
    "pop_2023",
    "weather",
    "vehicle",
    "drimpair",
    "driverrf",
    "distract",
    "cevent",
    "pbtype"
]

for table_name in tables:

    df = pd.read_csv(f"{table_name}.csv", low_memory=False)
    

    globals()[table_name] = df
    
    df.to_sql(table_name, conn, if_exists="replace", index=False)

In [16]:
# data model to be imported in power BI

data = pd.read_sql_query(
"""

SELECT

STATENAME AS "State",
REPLACE(p."2023", ',', '') AS "Population",
MONTHNAME AS "Month",
DAY_WEEKNAME AS "Day",
PEDS AS "Pedestrians",
FATALS AS "Fatalities"
FROM accident_project a
JOIN pop_2023 p
ON a.STATENAME = p.REGION

""",
conn)

In [17]:
data.head()

,State,Population,Month,Day,Pedestrians,Fatalities
0,Alabama,5108468,January,Wednesday,0,1
1,Alabama,5108468,March,Monday,0,1
2,Alabama,5108468,March,Saturday,0,1
3,Alabama,5108468,March,Friday,0,1
4,Alabama,5108468,March,Friday,0,1


In [18]:
df.head(1)

,STATE,STATENAME,ST_CASE,VEH_NO,PER_NO,PBAGE,PBAGENAME,PBSEX,PBSEXNAME,PBPTYPE,...,MOTMAN,MOTMANNAME,PEDLEG,PEDLEGNAME,PEDSNR,PEDSNRNAME,PEDCGP,PEDCGPNAME,BIKECGP,BIKECGPNAME
0,1,Alabama,10014,0,1,63,63 Years,1,Male,5,...,8,Not Applicable,8,Not Applicable,8,Not Applicable,400,Walking/Running Along Roadway,0,Not a Cyclist


In [7]:
# number of fatal accidents

num_of_accidents = pd.read_sql_query(
    
"""

SELECT
COUNT(*) AS "Fatal Accidents"
FROM accident_project 

"""
    , conn)

In [8]:
num_of_accidents

,Fatal Accidents
0,37654


In [9]:
# fatalities, accidents, fatalities per accident by state

fatals_accs_by_state = pd.read_sql_query(
    
"""

SELECT
STATENAME AS "State",
SUM(FATALS) AS "Fatalities",
COUNT(*) AS "Accidents",
ROUND((SUM(FATALS) * 1.0 / COUNT(*)),3) AS "Fatalities by Accident"
FROM accident_project 
GROUP BY STATENAME
ORDER BY COUNT(*) DESC
LIMIT 10

"""
    , conn)

In [10]:
fatals_accs_by_state.head()

,State,Fatalities,Accidents,Fatalities by Accident
0,Texas,4291,3874,1.108
1,California,4061,3727,1.090
2,Florida,3396,3183,1.067
3,Georgia,1615,1491,1.083
4,North Carolina,1561,1449,1.077


In [11]:
# accident frequency by month


fatals_accs_by_month = pd.read_sql_query(
    
"""

SELECT
MONTHNAME AS "Month",
COUNT(*) AS "Accidents",
SUM(FATALS) AS "Fatalities"
FROM accident_project 
GROUP BY MONTHNAME
ORDER BY Fatalities DESC

"""
    , conn)

In [12]:
fatals_accs_by_month.head()

,Month,Accidents,Fatalities
0,October,3505,3787
1,August,3423,3738
2,September,3409,3694
3,July,3401,3694
4,May,3229,3540


In [13]:
# Accidents by day of the week

fatals_accs_by_weekday = pd.read_sql_query(
    
"""

SELECT
DAY_WEEKNAME AS "Day",
COUNT(*) AS "Accidents",
SUM(FATALS) AS "Fatalities"
FROM accident_project 
GROUP BY DAY_WEEK

"""
    , conn)

In [14]:
fatals_accs_by_weekday

,Day,Accidents,Fatalities
0,Sunday,6033,6646
1,Monday,4859,5259
2,Tuesday,4584,4911
3,Wednesday,4832,5205
4,Thursday,4960,5357
5,Friday,5810,6306
6,Saturday,6576,7217


In [15]:
# rural vs urban number of accidents and fatalities


fatals_accs_by_rur_urb = pd.read_sql_query(
    
"""

SELECT
RUR_URBNAME AS "Rural/ Urban",
COUNT(*) AS "Accidents",
SUM(FATALS) AS "Fatalities"
FROM accident_project
WHERE RUR_URBNAME IN ('Rural', 'Urban')
GROUP BY RUR_URBNAME

"""
    , conn)

In [16]:
fatals_accs_by_rur_urb

,Rural/ Urban,Accidents,Fatalities
0,Rural,14938,16656
1,Urban,22400,23921


In [17]:
# number of accidents involving pedestrians


accs_involve_peds = pd.read_sql_query(
    
"""

SELECT
COUNT(*) AS "Accidents Involving Pedestrians"
FROM accident_project 
WHERE PEDS > 0
"""
    , conn)

In [18]:
accs_involve_peds

,Accidents Involving Pedestrians
0,8714


In [19]:
# Number of accidents (day vs night)


fatals_accs_by_light_cond = pd.read_sql_query(
    
"""

SELECT
LGT_CONDNAME AS "Light Condition",
COUNT(*) AS "Accidents",
SUM(FATALS) AS "Fatalities"
FROM accident_project 
WHERE LGT_CONDNAME NOT IN ('Other', 'Not Reported', 'Reported as Unknown', 'Dark - Unknown Lighting')
GROUP BY LGT_CONDNAME
ORDER BY COUNT(*) DESC

"""
    , conn)

In [20]:
fatals_accs_by_light_cond

,Light Condition,Accidents,Fatalities
0,Daylight,16826,18298
1,Dark - Not Lighted,10221,11165
2,Dark - Lighted,8176,8805
3,Dusk,983,1064
4,Dawn,725,791


In [172]:
# top 5 hours for number of accidents


top_5_hours_fatals_accs = pd.read_sql_query(
    
"""

SELECT
HOUR || ":00" AS "Hour",
COUNT(*) AS "Accidents",
SUM(FATALS) AS "Fatalities"
FROM accident_project 
WHERE HOUR <> "99"
GROUP BY HOUR
ORDER BY COUNT(*) DESC
LIMIT 5

"""
    , conn)

In [173]:
top_5_hours_fatals_accs

,Hour,Accidents,Fatalities
0,20:00,2307,2501
1,21:00,2280,2459
2,18:00,2237,2387
3,19:00,2194,2354
4,17:00,2099,2267


In [174]:
# Accidents, fatalities, and weighted fatalities in weekends vs weekdays


weighted_fatals_weekend_weekday = pd.read_sql_query(
    
"""

SELECT
CASE 
    WHEN DAY_WEEKNAME IN ('Sunday', 'Saturday') THEN "Weekend"
    ELSE "Workday" 
END AS "Day Week Category",
COUNT(*) AS "Accidents",
SUM(FATALS) AS "Fatalities",
CASE
    WHEN "DAY WEEK CATEGORY" = "Weekend" THEN (SUM(FATALS) * 2) / 7
    ELSE (SUM(FATALS) * 5) / 7 END AS "Weighted Fatalities"
FROM accident_project 
GROUP BY "Day Week Category"


"""
    , conn)

In [175]:
weighted_fatals_weekend_weekday

,Day Week Category,Accidents,Fatalities,Weighted Fatalities
0,Weekend,12609,13863,9902
1,Workday,25045,27038,19312


In [176]:
# state fatalities per 100k population

st_fatals_per_100k = pd.read_sql_query(
    
"""

SELECT
a.STATENAME AS "State",
p."2023" AS "Population_2023",
SUM(a.FATALS) AS "Fatalities",
ROUND(SUM(a.FATALS * 1.0) / (CAST(REPLACE(MAX(p."2023"), ',', '') AS INTEGER) / 100000), 2) AS Fatalities_per_100k
FROM accident_project a
JOIN pop_2023 p
ON a.STATENAME = p.Region
GROUP BY a.STATENAME
ORDER BY Fatalities_per_100k desc

"""
    , conn)

In [178]:
st_fatals_per_100k.head()

,State,Population_2023,Fatalities,Fatalities_per_100k
0,Wyoming,"584,057",144,28.80
1,Mississippi,"2,939,690",732,25.24
2,New Mexico,"2,114,371",437,20.81
3,Arkansas,"3,067,732",596,19.87
4,South Carolina,"5,373,555",1047,19.75


In [183]:
fatals_accs_by_weather = pd.read_sql_query(


"""

SELECT
w.WEATHERNAME AS "Weather",
COUNT(a.FATALS) AS "Accidents",
SUM(a.FATALS) AS "Fatalities"
FROM 
accident_project a
JOIN weather w
ON a.ST_CASE = w.ST_CASE
WHERE w.WEATHERNAME NOT IN ('Not Reported', 'Reported as Unknown')
GROUP BY w.WEATHERNAME
ORDER BY "Accidents" DESC

"""
 
, conn)

In [184]:
fatals_accs_by_weather.head()

,Weather,Accidents,Fatalities
0,Clear,27872,30313
1,Cloudy,5202,5629
2,Rain,2663,2867
3,"Fog, Smog, Smoke",420,472
4,Snow,222,247


In [188]:
# difference between arrival time and notification time
# difference between hospital time and notification time

# Note: removing states where count is 30 or less due to lack of reporting making data unreliable


EMS_response_time_by_state = pd.read_sql_query(
    
"""

SELECT
STATENAME AS "State",
ROUND(AVG((ARR_HOUR * 60 + ARR_MIN - (NOT_HOUR * 60 + NOT_MIN) + 1440) % 1440), 2) AS "EMS Notification to EMS Arrival Time",
ROUND(AVG((HOSP_HR * 60 + HOSP_MN - (ARR_HOUR * 60 + ARR_MIN) + 1440) % 1440), 2) AS "EMS Arrival to Hospital Arrival Time"
FROM accident_project
WHERE NOT_HOUR BETWEEN 0 AND 23
AND ARR_HOUR BETWEEN 0 AND 23
AND HOSP_HR BETWEEN 0 AND 23
AND NOT_MIN BETWEEN 0 AND 59
AND ARR_MIN BETWEEN 0 AND 59
AND HOSP_MN BETWEEN 0 AND 59
GROUP BY STATENAME
HAVING COUNT(*) > 30


"""


    , conn)

In [189]:
EMS_response_time_by_state.head()

,State,EMS Notification to EMS Arrival Time,EMS Arrival to Hospital Arrival Time
0,Alabama,16.18,36.52
1,Arizona,15.64,33.68
2,Arkansas,10.24,35.29
3,Colorado,6.99,23.34
4,Connecticut,7.60,30.99


In [192]:
# accidents and fatalities by vehicle make

top_10_veh_make_fatals_accs = pd.read_sql_query(

"""
SELECT
v.VPICMAKENAME AS "Vehicle Make",
COUNT(*) AS "Accidents",
SUM(a.FATALS) AS "Fatalities"
FROM accident_project a
JOIN vehicle v 
ON a.ST_CASE = v.ST_CASE
GROUP BY v.VPICMAKENAME
ORDER BY "Fatalities" DESC
LIMIT 10

"""
 
, conn)

In [193]:
top_10_veh_make_fatals_accs

,Vehicle Make,Accidents,Fatalities
0,Chevrolet,7261,8132
1,Ford,7150,7946
2,Toyota,5161,5824
3,Honda,4453,4878
4,Nissan,3279,3680
5,Dodge,2419,2723
6,Harley Davidson,2271,2379
7,Jeep,1742,1937
8,GMC,1732,1914
9,Hyundai,1606,1810


In [196]:
# accidents and fatalities by hit and run


hit_run_fatals_accs = pd.read_sql_query(

"""
SELECT 
HIT_RUNNAME AS "Hit and Run",
COUNT(*) AS "Accidents",
SUM(DEATHS) AS "Fatalities",
ROUND(SUM(DEATHS * 1.0) / COUNT(*), 2) AS "Avg Fatalities"
FROM vehicle
GROUP BY HIT_RUNNAME

"""
    
, conn)

In [197]:
hit_run_fatals_accs

,Hit and Run,Accidents,Fatalities,Avg Fatalities
0,No,55450,31929,0.58
1,Yes,2869,152,0.05


In [202]:
# hit and run AND under the influence

hit_run_impaired_fatals_accs = pd.read_sql_query(


"""
SELECT
d.DRIMPAIRNAME AS "Hit & Run Impairment",
COUNT(*) AS "Accidents",
SUM(v.DEATHS) AS "Fatalities"
FROM vehicle v
LEFT JOIN drimpair d
ON v.ST_CASE = d.ST_CASE
WHERE v.HIT_RUN = "1" AND d.DRIMPAIRNAME NOT IN ("Reported as Unknown if Impaired", "Not Reported")
GROUP BY d.DRIMPAIRNAME
ORDER BY Accidents DESC

"""

    
, conn)

In [203]:
hit_run_impaired_fatals_accs

,Hit & Run Impairment,Accidents,Fatalities
0,None/Apparently Normal,726,34
1,"Under the Influence of Alcohol, Drugs or Medic...",337,41
2,No Driver Present/Unknown if Driver Present,28,1
3,"Emotional (depressed, angry, disturbed, etc.)",15,3
4,Asleep or Fatigued,9,0
5,Physical Impairment - No Details,8,0
6,Other Physical Impairment,4,0
7,"Ill, Blackout",4,0
8,Blind/Low Vision,2,0


In [208]:
driver_crash_cause = pd.read_sql_query(

"""
SELECT
d.DRIVERRFNAME AS "Driver Crash Related Factors",
COUNT(*) AS "Accidents",
SUM(a.FATALS) AS "Fatalities"
FROM accident_project a
JOIN driverrf d
ON a.ST_CASE = d.ST_CASE
WHERE "Driver Crash Related Factors" NOT IN ('None Noted')
GROUP BY d.DRIVERRFNAME
ORDER BY Accidents DESC
LIMIT 5
"""
    
, conn)

In [209]:
driver_crash_cause

,Driver Crash Related Factors,Accidents,Fatalities
0,"Careless Driving, Inattentive Operation, Impro...",4781,5299
1,Failure to Yield Right-of-Way,4584,4952
2,Improper Lane Usage,3230,3798
3,"Failure to Obey Actual Traffic Signs, Traffic ...",2523,2855
4,"Operating the Vehicle in an Erratic, Reckless ...",2172,2446


In [210]:
crash_seq_of_event_breakdown = pd.read_sql_query(

"""

SELECT
c.SOENAME AS "Sequence of Events",
COUNT(*) AS "Accidents",
SUM(a.FATALS) AS "Fatalities",
ROUND(SUM(a.FATALS * 1.0) / COUNT(*), 2) AS "Avg Fatalities"
FROM accident_project a
JOIN cevent c
ON a.ST_CASE = c.ST_CASE
GROUP BY c.SOENAME
ORDER BY Accidents DESC
LIMIT 5

"""
, conn)

In [211]:
crash_seq_of_event_breakdown

,Sequence of Events,Accidents,Fatalities,Avg Fatalities
0,Motor Vehicle In-Transport,19599,22674,1.16
1,Ran Off Roadway - Right,12191,13453,1.10
2,Ran Off Roadway - Left,9048,9984,1.10
3,Rollover/Overturn,8698,9707,1.12
4,Pedestrian,8279,8739,1.06


In [212]:
fatal_pedestrian_gender = pd.read_sql_query(
"""

SELECT
p.PBSEXNAME AS "Gender",
COUNT(*) AS "Pedestrian Fatalities"
FROM accident_project a
JOIN pbtype p
ON a.ST_CASE = p.ST_CASE
WHERE PBSEXNAME IN ('Female', 'Male')
GROUP BY p.PBSEXNAME


"""
, conn)

In [213]:
fatal_pedestrian_gender

,Gender,Pedestrian Fatalities
0,Female,2479
1,Male,6644


In [45]:
fatal_pedestrian_age_range = pd.read_sql_query(
"""

SELECT
CASE WHEN p.PBAGE BETWEEN 0 AND 1 THEN "0-1"
WHEN p.PBAGE BETWEEN 2 AND 4 THEN "2-4"
WHEN p.PBAGE BETWEEN 5 AND 9 THEN "5-9"
WHEN p.PBAGE BETWEEN 10 AND 17 THEN "10-17"
WHEN p.PBAGE BETWEEN 18 AND 25 THEN "18-24"
WHEN p.PBAGE BETWEEN 25 AND 34 THEN "25-34"
WHEN p.PBAGE BETWEEN 35 AND 44 THEN "35-44"
WHEN p.PBAGE BETWEEN 45 AND 54 THEN "45-54"
WHEN p.PBAGE BETWEEN 55 AND 64 THEN "55-64"
ELSE "65+" END AS "Age Range",
COUNT(*) AS "Pedestrian Fatalities"
FROM accident_project a
JOIN pbtype p
ON a.ST_CASE = p.ST_CASE
WHERE p.PBAGE NOT IN ('999', '998')
GROUP BY "Age Range"
ORDER BY "Pedestrian Fatalities" DESC


"""
, conn)

In [46]:
fatal_pedestrian_age

,Age Range,Pedestrian Fatalities
0,65+,1853
1,35-44,1623
2,55-64,1614
3,45-54,1379
4,25-34,1334
5,18-24,788
6,10-17,310
7,5-9,69
8,2-4,58
9,0-1,27
